# Træn v2_robust YOLOv11s — FCM Landmark Detection

Mixed-conditions træning: hvert billede får tilfældig støj, frø-størrelse og baggrund.
**Ingen plade-imperfektioner** — det kommer i v3.

Output: `models/v2_robust/best.pt`.

## Sådan kører du

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (Cmd/Ctrl + F9)
3. ~5-10 min træning
4. Sidste celle downloader `best.pt`
5. På din Mac:
   ```bash
   mkdir -p models/v2_robust
   mv ~/Downloads/best.pt models/v2_robust/
   fcm-switch v2_robust
   ```

## 1. Setup

In [ ]:
!pip install ultralytics==8.4.47 -q

import ultralytics, torch
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch:     {torch.__version__}')
print(f'GPU:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "INGEN — skift runtime til T4!"}')

## 2. Datagenerator (mixed conditions)

Hvert billede: tilfældig støj-niveau, frø-størrelse, baggrunds-reflektans.

In [ ]:
import numpy as np
from pathlib import Path
import shutil
from PIL import Image

NOISE_LEVELS = [0.02, 0.05, 0.10]
SEED_SIZE_PRESETS = [
    (8, 12, 3, 5),    # mindre
    (12, 18, 5, 7),   # normal
    (18, 24, 7, 9),   # større
]
BACKGROUND_REFLECTANCE_MIN = 0.75
BACKGROUND_REFLECTANCE_MAX = 0.95
SEED_REFLECTANCE_MIN, SEED_REFLECTANCE_MAX = 0.20, 0.45
PACKING_GAP_PIXELS = 2
MAX_PLACEMENT_ATTEMPTS = 200


def ellipses_overlap(e1, e2, gap):
    cx1, cy1, L1, _, _ = e1
    cx2, cy2, L2, _, _ = e2
    r1, r2 = L1 / 2.0, L2 / 2.0
    dx, dy = cx1 - cx2, cy1 - cy2
    return np.sqrt(dx*dx + dy*dy) < (r1 + r2 + gap)


def draw_ellipse(image, cx, cy, length, width, angle_deg, color):
    a, b = length / 2.0, width / 2.0
    rad = np.deg2rad(angle_deg)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    H, W = image.shape
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    xs, ys = xx - cx, yy - cy
    xr = cos_a * xs + sin_a * ys
    yr = -sin_a * xs + cos_a * ys
    image[(xr/a)**2 + (yr/b)**2 < 1.0] = color


def generate_one_image(rng, num_seeds, img_size=256):
    """Mixed-conditions billede — tilfældig støj, frø-størrelse, baggrund."""
    noise_level = float(rng.choice(NOISE_LEVELS))
    L_min, L_max, W_min, W_max = SEED_SIZE_PRESETS[rng.integers(0, 3)]
    bg = float(rng.uniform(BACKGROUND_REFLECTANCE_MIN, BACKGROUND_REFLECTANCE_MAX))

    img = np.ones((img_size, img_size), dtype=np.float32) * bg
    placed, centers = [], []
    margin = L_max // 2 + 2

    for _ in range(num_seeds):
        L = int(rng.integers(L_min, L_max + 1))
        W = int(rng.integers(W_min, W_max + 1))
        for _ in range(MAX_PLACEMENT_ATTEMPTS):
            cx = int(rng.integers(margin, img_size - margin))
            cy = int(rng.integers(margin, img_size - margin))
            angle = float(rng.uniform(0, 360))
            new_e = (cx, cy, L, W, angle)
            if not any(ellipses_overlap(new_e, ex, PACKING_GAP_PIXELS) for ex in placed):
                color = float(rng.uniform(SEED_REFLECTANCE_MIN, SEED_REFLECTANCE_MAX))
                draw_ellipse(img, cx, cy, L, W, angle, color)
                placed.append(new_e)
                centers.append((cx, cy))
                break

    noise = rng.standard_normal((img_size, img_size)) * noise_level
    img = np.clip(img + noise, 0.0, 1.0)
    return (img * 255).astype(np.uint8), centers, noise_level

print('Generator klar.')

## 3. Generér datasæt

1200 billeder, 80/20 split.

In [ ]:
NUM_IMAGES = 1200
IMG_SIZE = 256
BBOX_SIZE = 18
TRAIN_SPLIT = 0.8

dataset_dir = Path('dataset')
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)
for split in ('train', 'val'):
    (dataset_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dataset_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
indices = set(rng.permutation(NUM_IMAGES)[:int(NUM_IMAGES * TRAIN_SPLIT)].tolist())

noise_distribution = []
for i in range(NUM_IMAGES):
    n_seeds = int(rng.integers(40, 81))
    img, centers, noise = generate_one_image(rng, n_seeds, IMG_SIZE)
    noise_distribution.append(noise)
    split = 'train' if i in indices else 'val'

    Image.fromarray(img).save(dataset_dir / 'images' / split / f'img_{i:04d}.png')
    with open(dataset_dir / 'labels' / split / f'img_{i:04d}.txt', 'w') as f:
        for (cx, cy) in centers:
            x_n, y_n = cx / IMG_SIZE, cy / IMG_SIZE
            s_n = BBOX_SIZE / IMG_SIZE
            f.write(f'0 {x_n:.6f} {y_n:.6f} {s_n:.6f} {s_n:.6f}\n')

    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{NUM_IMAGES} billeder')

(dataset_dir / 'data.yaml').write_text('train: images/train\nval: images/val\nnc: 1\nnames:\n  0: seed\n')

from collections import Counter
print(f'\nDatasæt klar: {len(indices)} train, {NUM_IMAGES - len(indices)} val')
print(f'Noise-distribution: {dict(Counter(round(n, 3) for n in noise_distribution))}')

## 4. Træning på T4 GPU

Standard augmentations — flips, rotation. **Ingen** mixup/copy_paste/perspective.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data='dataset/data.yaml',
    epochs=50,
    imgsz=256,
    batch=32,
    patience=10,
    project='runs',
    name='v2_robust',
    exist_ok=True,
    device=0,
    verbose=True,
    flipud=0.5,
    fliplr=0.5,
    degrees=180.0,
)

print('\n=== TRÆNING FÆRDIG ===')
print(f'Precision: {results.box.mp:.4f}')
print(f'Recall:    {results.box.mr:.4f}')
print(f'mAP@50:    {results.box.map50:.4f}')
print(f'mAP@50-95: {results.box.map:.4f}')

## 5. Download best.pt

```bash
mkdir -p models/v2_robust
mv ~/Downloads/best.pt models/v2_robust/
fcm-switch v2_robust
fcm-demo
```

In [ ]:
import glob
from pathlib import Path
from google.colab import files

matches = glob.glob('**/best.pt', recursive=True)
if not matches:
    raise FileNotFoundError('Ingen best.pt fundet')
best_pt = matches[0]
size_mb = Path(best_pt).stat().st_size / 1024 / 1024
print(f'Model: {best_pt} ({size_mb:.2f} MB)')
files.download(best_pt)